# Mini Project 1: Indoor Localization Using Wi-Fi RSSI Fingerprints
**Topic:** Telecommunications | **Course:** Machine Learning Fundamentals  
**Dataset:** UJIIndoorLoc (`trainingData.csv`, `validationData.csv`)  
**Purpose:** Reproducible analysis for classification (`BUILDINGID`, `FLOOR`) and regression (`LONGITUDE`, `LATITUDE`).

> Run from top to bottom. This notebook deliberately does not hard-code performance results. Results are computed from the uploaded CSV files. The provided validation file is reserved for additional final evaluation; model selection uses training data only.


## 1. Setup and reproducibility
The assignment requires `random_state=42`, 80:20 and 70:30 splits, stratification for classification, and 5-fold CV on training folds. Imputers and scalers are fitted only inside pipelines. `+100` means a WAP was not detected and is replaced with `-105 dBm` as the physical-value strategy specified in the assignment brief. A second strategy is included for comparison: convert `+100` to missing and median-impute within a pipeline.

In [1]:
from pathlib import Path
import time, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, mean_absolute_error,
    mean_squared_error, r2_score, silhouette_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_DIR = Path(".")  # Put both CSV files beside this notebook, or update this path.
TRAIN_PATH = DATA_DIR / "trainingData.csv"
VALIDATION_PATH = DATA_DIR / "validationData.csv"

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)


ModuleNotFoundError: No module named 'pandas'

In [ ]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)

rssi_cols = [c for c in train.columns if c.startswith("WAP")]
target_cols = ["BUILDINGID", "FLOOR", "LONGITUDE", "LATITUDE"]
metadata_cols = ["SPACEID", "RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"]
required = set(rssi_cols + target_cols)
missing_required = required.difference(train.columns)
assert not missing_required, f"Missing expected columns: {missing_required}"
assert set(rssi_cols).issubset(validation.columns)

print("Training shape:", train.shape)
print("Validation shape:", validation.shape)
print("RSSI features:", len(rssi_cols))
print("Training missing cells:", int(train.isna().sum().sum()))
print("Validation missing cells:", int(validation.isna().sum().sum()))
print("Training buildings:", sorted(train.BUILDINGID.unique()))
print("Training floors:", sorted(train.FLOOR.unique()))
display(train[target_cols].describe().T)


## 2. Exploratory data analysis

In [ ]:
# Label distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
train["BUILDINGID"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Building distribution"); axes[0].set_xlabel("BUILDINGID"); axes[0].set_ylabel("Samples")
train["FLOOR"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Floor distribution"); axes[1].set_xlabel("FLOOR"); axes[1].set_ylabel("Samples")
plt.tight_layout(); plt.show()

train_rssi = train[rssi_cols]
val_rssi = validation[rssi_cols]
print("Training +100 count:", int((train_rssi == 100).sum().sum()))
print("Validation +100 count:", int((val_rssi == 100).sum().sum()))
print("Training +100 percent:", round((train_rssi == 100).to_numpy().mean()*100, 3))
print("Validation +100 percent:", round((val_rssi == 100).to_numpy().mean()*100, 3))

active_counts = (train_rssi != 100).sum(axis=0)
print("WAP never detected in training:", int((active_counts == 0).sum()))
print("WAP detected in <=10 training rows:", int((active_counts <= 10).sum()))
display(active_counts.sort_values().head(15).rename("detected_rows").to_frame())
display(active_counts.sort_values(ascending=False).head(15).rename("detected_rows").to_frame())

# RSSI distribution for detected signal values only
detected_values = train_rssi.where(train_rssi != 100).stack()
plt.figure(figsize=(8, 4))
plt.hist(detected_values, bins=40)
plt.title("RSSI distribution for detected WAPs only")
plt.xlabel("RSSI (dBm)"); plt.ylabel("Cell count")
plt.tight_layout(); plt.show()


## 3. Preprocessing strategies and leakage-safe feature construction
- **Strategy A:** replace `+100` with `-105 dBm`.
- **Strategy B:** replace `+100` with `NaN`, then median-impute inside a pipeline.
- WAP columns that are never detected in training can be removed, with this decision explicitly documented.
- Metadata/ID fields are excluded from position predictors.
- Scaling is fitted inside each pipeline on training folds only.


In [ ]:
def make_X(frame, strategy="physical", drop_never_detected=True, reference_train=None):
    X = frame[rssi_cols].copy()
    if strategy == "physical":
        X = X.replace(100, -105.0)
    elif strategy == "median":
        X = X.replace(100, np.nan)
    else:
        raise ValueError("strategy must be 'physical' or 'median'")
    if drop_never_detected:
        ref = reference_train if reference_train is not None else train
        ref_active = (ref[rssi_cols] != 100).sum(axis=0)
        keep = ref_active[ref_active > 0].index.tolist()
        X = X[keep]
    return X

def make_pipeline(estimator, strategy="physical", scale=True):
    steps = []
    if strategy == "median":
        steps.append(("imputer", SimpleImputer(strategy="median")))
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)

# Compare preprocessing summaries, not model performance yet.
preprocess_summary = pd.DataFrame([
    {"strategy":"Physical replacement (-105 dBm)", "train_features":len(make_X(train, "physical").columns),
     "train_nan_after_transform":int(make_X(train, "physical").isna().sum().sum())},
    {"strategy":"Missing + median imputation in Pipeline", "train_features":len(make_X(train, "median").columns),
     "train_nan_before_pipeline":int(make_X(train, "median").isna().sum().sum())}
])
display(preprocess_summary)


## 4. Clustering: K-Means and DBSCAN
Clustering is exploratory. Scale the RSSI data after physical replacement, use elbow/inertia and silhouette to choose a candidate K, then assess DBSCAN using a k-distance curve. For runtime, clustering can be run on a reproducible sample. ARI/NMI compare clusters with `BUILDINGID` and `FLOOR`; labels are used for evaluation only, not as clustering inputs.

In [ ]:
# Use physical replacement and remove WAPs never detected in training.
X_cluster = make_X(train, "physical", drop_never_detected=True)
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

rng = np.random.RandomState(RANDOM_STATE)
sample_n = min(5000, len(train))
sample_idx = rng.choice(len(train), size=sample_n, replace=False)
Xc = X_cluster_scaled[sample_idx]
cluster_labels_building = train["BUILDINGID"].to_numpy()[sample_idx]
cluster_labels_floor = train["FLOOR"].to_numpy()[sample_idx]

inertias, silhouettes = [], []
candidate_ks = range(2, 9)
for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(Xc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Xc, labels, sample_size=min(2000, len(Xc)), random_state=RANDOM_STATE))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(candidate_ks), inertias, marker="o"); axes[0].set_title("K-Means elbow"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(candidate_ks), silhouettes, marker="o"); axes[1].set_title("K-Means silhouette"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()
display(pd.DataFrame({"k":list(candidate_ks), "inertia":inertias, "silhouette":silhouettes}))


In [ ]:
# DBSCAN k-distance plot. This can be computationally demanding for large datasets.
from sklearn.neighbors import NearestNeighbors
k_dist = 5
nn = NearestNeighbors(n_neighbors=k_dist).fit(Xc)
distances, _ = nn.kneighbors(Xc)
kdist_sorted = np.sort(distances[:, -1])
plt.figure(figsize=(7, 4)); plt.plot(kdist_sorted)
plt.title(f"DBSCAN {k_dist}-distance plot (sample)")
plt.xlabel("Samples sorted by k-distance"); plt.ylabel("Distance"); plt.tight_layout(); plt.show()

# Set eps after visually inspecting the plot. This is an explicit experiment parameter.
DBSCAN_EPS = float(np.quantile(kdist_sorted, 0.90))  # initial reproducible candidate; test nearby values too
db = DBSCAN(eps=DBSCAN_EPS, min_samples=5).fit_predict(Xc)
valid_mask = db != -1
print("DBSCAN eps candidate:", DBSCAN_EPS)
print("Noise fraction:", round(float((db == -1).mean()), 4))
if len(set(db[valid_mask])) >= 2 and valid_mask.sum() > 2:
    print("Silhouette excluding noise:", silhouette_score(Xc[valid_mask], db[valid_mask], sample_size=min(2000, int(valid_mask.sum())), random_state=RANDOM_STATE))
else:
    print("Silhouette unavailable: fewer than two non-noise clusters.")
print("ARI building:", adjusted_rand_score(cluster_labels_building, db))
print("NMI building:", normalized_mutual_info_score(cluster_labels_building, db))
print("ARI floor:", adjusted_rand_score(cluster_labels_floor, db))
print("NMI floor:", normalized_mutual_info_score(cluster_labels_floor, db))

# PCA 2D visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(Xc)
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=cluster_labels_building, s=8)
plt.title("PCA 2D coloured by true BUILDINGID (for interpretation only)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout(); plt.show()
print("PCA explained variance ratio:", pca.explained_variance_ratio_)


## 5. Supervised model experiments
Classification targets: `BUILDINGID` and `FLOOR`. Regression targets: `LONGITUDE` and `LATITUDE`. The six baseline models are Logistic Regression, KNN, Decision Tree, SVM, Random Forest, and HistGradientBoosting. Regression baselines use Linear Regression, Ridge, Decision Tree, SVR, Random Forest, and HistGradientBoosting. Five-fold CV runs only on the training partition. To keep runtime manageable, set `RUN_FULL_CV=True` when running the complete matrix.

In [ ]:
RUN_FULL_CV = True
SPLITS = {"80:20": 0.20, "70:30": 0.30}
CV_FOLDS = 5

def classification_metrics(y_true, y_pred, y_prob=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob[:, 1])
    return out

def regression_metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred)**0.5,
            "R2": r2_score(y_true, y_pred)}

classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}
regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "SVR": SVR(C=1.0, epsilon=0.1),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}


In [ ]:
# Full experiment matrix. Classification uses stratified splits and StratifiedKFold.
# Regression uses random splits and KFold. Data validation file remains untouched here.
results = []
best_models = {}

for split_name, test_size in SPLITS.items():
    for strategy in ["physical", "median"]:
        X_all = make_X(train, strategy=strategy, drop_never_detected=True)
        for target in ["BUILDINGID", "FLOOR"]:
            y_all = train[target]
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_all, y_all, test_size=test_size, random_state=RANDOM_STATE, stratify=y_all
            )
            cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
            for model_name, estimator in classification_models.items():
                pipe = make_pipeline(estimator, strategy=strategy, scale=(model_name in ["Logistic Regression","KNN","SVM"]))
                start = time.time()
                if RUN_FULL_CV:
                    scores = cross_validate(pipe, X_tr, y_tr, cv=cv,
                        scoring={"accuracy":"accuracy","precision_macro":"precision_macro",
                                 "recall_macro":"recall_macro","f1_macro":"f1_macro"},
                        n_jobs=-1, return_train_score=False)
                    cv_f1_mean = scores["test_f1_macro"].mean()
                    cv_f1_std = scores["test_f1_macro"].std()
                else:
                    cv_f1_mean, cv_f1_std = np.nan, np.nan
                pipe.fit(X_tr, y_tr)
                pred = pipe.predict(X_te)
                row = {"split":split_name,"preprocessing":strategy,"task":"classification",
                       "target":target,"model":model_name,"cv_f1_macro_mean":cv_f1_mean,
                       "cv_f1_macro_std":cv_f1_std, **classification_metrics(y_te,pred),
                       "fit_and_eval_seconds":time.time()-start}
                results.append(row)
                best_models[(split_name,strategy,target,model_name)] = pipe
                print("Done:", split_name, strategy, target, model_name)

        for target in ["LONGITUDE", "LATITUDE"]:
            y_all = train[target]
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_all, y_all, test_size=test_size, random_state=RANDOM_STATE
            )
            cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
            for model_name, estimator in regression_models.items():
                pipe = make_pipeline(estimator, strategy=strategy, scale=(model_name in ["Ridge","SVR"]))
                start = time.time()
                if RUN_FULL_CV:
                    scores = cross_validate(pipe, X_tr, y_tr, cv=cv,
                        scoring={"mae":"neg_mean_absolute_error","rmse":"neg_root_mean_squared_error","r2":"r2"},
                        n_jobs=-1, return_train_score=False)
                    cv_mae_mean = -scores["test_mae"].mean()
                    cv_mae_std = scores["test_mae"].std()
                else:
                    cv_mae_mean, cv_mae_std = np.nan, np.nan
                pipe.fit(X_tr, y_tr)
                pred = pipe.predict(X_te)
                row = {"split":split_name,"preprocessing":strategy,"task":"regression",
                       "target":target,"model":model_name,"cv_MAE_mean":cv_mae_mean,
                       "cv_MAE_std":cv_mae_std, **regression_metrics(y_te,pred),
                       "fit_and_eval_seconds":time.time()-start}
                results.append(row)
                best_models[(split_name,strategy,target,model_name)] = pipe
                print("Done:", split_name, strategy, target, model_name)

results_df = pd.DataFrame(results)
display(results_df.sort_values(["task","target","split","preprocessing"]))
results_df.to_csv("supervised_model_results.csv", index=False)


## 6. Confusion matrices and regression diagnostic plots
Select models using cross-validation results on training data. Do not choose the winner based on the reserved validation file. The held-out split is reported separately. Regression coordinates in this dataset appear to be projected map coordinates despite the column names; verify the dataset documentation before converting them to meters. Do not interpret raw coordinate MAE as meters without verification.

In [ ]:
# Example diagnostics for best model by macro-F1 and MAE within each target/split/preprocessing.
if not results_df.empty:
    cls = results_df[results_df.task == "classification"]
    for (split_name, strategy, target), group in cls.groupby(["split","preprocessing","target"]):
        winner = group.sort_values("cv_f1_macro_mean", ascending=False).iloc[0]
        model = best_models[(split_name,strategy,target,winner["model"])]
        # Recreate same split for diagnostic plot
        X_all = make_X(train, strategy=strategy, drop_never_detected=True)
        X_tr, X_te, y_tr, y_te = train_test_split(X_all, train[target],
            test_size=SPLITS[split_name], random_state=RANDOM_STATE, stratify=train[target])
        pred = model.predict(X_te)
        ConfusionMatrixDisplay.from_predictions(y_te, pred, xticks_rotation="vertical")
        plt.title(f"{target} | {split_name} | {strategy} | {winner['model']}")
        plt.tight_layout(); plt.show()

    reg = results_df[results_df.task == "regression"]
    for (split_name, strategy, target), group in reg.groupby(["split","preprocessing","target"]):
        winner = group.sort_values("cv_MAE_mean", ascending=True).iloc[0]
        model = best_models[(split_name,strategy,target,winner["model"])]
        X_all = make_X(train, strategy=strategy, drop_never_detected=True)
        X_tr, X_te, y_tr, y_te = train_test_split(X_all, train[target],
            test_size=SPLITS[split_name], random_state=RANDOM_STATE)
        pred = model.predict(X_te)
        plt.figure(figsize=(5,5)); plt.scatter(y_te, pred, s=8)
        lo, hi = min(y_te.min(), pred.min()), max(y_te.max(), pred.max())
        plt.plot([lo,hi],[lo,hi])
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"{target} | {split_name} | {strategy} | {winner['model']}")
        plt.tight_layout(); plt.show()
        residual = y_te.to_numpy() - pred
        plt.figure(figsize=(6,3)); plt.scatter(pred, residual, s=8); plt.axhline(0)
        plt.xlabel("Predicted"); plt.ylabel("Residual")
        plt.title(f"Residual plot: {target} | {winner['model']}")
        plt.tight_layout(); plt.show()


## 7. MLP / ANN experiments
At least three architectures and at least two learning rates are required. This section tests three architectures for each target and two learning rates. Training curves are recorded from the fitted MLP. Use the same split definitions and compare with the best supervised baseline. For regression targets, interpret coordinate errors using the documented coordinate reference system.

In [ ]:
MLP_ARCHITECTURES = {
    "MLP_small": (128,),
    "MLP_medium": (256, 128),
    "MLP_deep": (256, 128, 64),
}
LEARNING_RATES = [0.001, 0.01]
mlp_results = []

for split_name, test_size in SPLITS.items():
    for strategy in ["physical", "median"]:
        X_all = make_X(train, strategy=strategy, drop_never_detected=True)
        for target in ["BUILDINGID", "FLOOR"]:
            y_all = train[target]
            X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=test_size,
                random_state=RANDOM_STATE, stratify=y_all)
            for arch_name, hidden_layers in MLP_ARCHITECTURES.items():
                for lr in LEARNING_RATES:
                    model = make_pipeline(MLPClassifier(hidden_layer_sizes=hidden_layers,
                        learning_rate_init=lr, max_iter=150, early_stopping=True,
                        validation_fraction=0.15, n_iter_no_change=12,
                        random_state=RANDOM_STATE), strategy=strategy, scale=True)
                    start=time.time(); model.fit(X_tr,y_tr); pred=model.predict(X_te)
                    inner = model.named_steps["model"]
                    mlp_results.append({"split":split_name,"preprocessing":strategy,"task":"classification",
                        "target":target,"architecture":arch_name,"hidden_layers":str(hidden_layers),
                        "learning_rate":lr,"accuracy":accuracy_score(y_te,pred),
                        "precision_macro":precision_score(y_te,pred,average="macro",zero_division=0),
                        "recall_macro":recall_score(y_te,pred,average="macro",zero_division=0),
                        "f1_macro":f1_score(y_te,pred,average="macro",zero_division=0),
                        "parameter_count":int(sum(np.prod(w.shape) for w in inner.coefs_)+sum(np.prod(b.shape) for b in inner.intercepts_)),
                        "epochs":inner.n_iter_,"fit_seconds":time.time()-start})
                    plt.figure(figsize=(5,3)); plt.plot(inner.loss_curve_, label="training loss")
                    if hasattr(inner,"validation_scores_") and inner.validation_scores_:
                        plt.title(f"{target} {arch_name} lr={lr}: loss curve (validation score also available)")
                    else: plt.title(f"{target} {arch_name} lr={lr}: training loss")
                    plt.xlabel("Iteration"); plt.ylabel("Loss"); plt.tight_layout(); plt.show()
        for target in ["LONGITUDE","LATITUDE"]:
            y_all=train[target]
            X_tr, X_te, y_tr, y_te=train_test_split(X_all,y_all,test_size=test_size,random_state=RANDOM_STATE)
            for arch_name, hidden_layers in MLP_ARCHITECTURES.items():
                for lr in LEARNING_RATES:
                    model=make_pipeline(MLPRegressor(hidden_layer_sizes=hidden_layers,
                        learning_rate_init=lr,max_iter=150,early_stopping=True,
                        validation_fraction=0.15,n_iter_no_change=12,random_state=RANDOM_STATE),
                        strategy=strategy,scale=True)
                    start=time.time(); model.fit(X_tr,y_tr); pred=model.predict(X_te)
                    inner=model.named_steps["model"]
                    mlp_results.append({"split":split_name,"preprocessing":strategy,"task":"regression",
                        "target":target,"architecture":arch_name,"hidden_layers":str(hidden_layers),
                        "learning_rate":lr,**regression_metrics(y_te,pred),
                        "parameter_count":int(sum(np.prod(w.shape) for w in inner.coefs_)+sum(np.prod(b.shape) for b in inner.intercepts_)),
                        "epochs":inner.n_iter_,"fit_seconds":time.time()-start})
                    plt.figure(figsize=(5,3)); plt.plot(inner.loss_curve_)
                    plt.title(f"{target} {arch_name} lr={lr}: training loss")
                    plt.xlabel("Iteration"); plt.ylabel("Loss"); plt.tight_layout(); plt.show()

mlp_results_df=pd.DataFrame(mlp_results)
display(mlp_results_df)
mlp_results_df.to_csv("mlp_results.csv",index=False)


## 8. Final additional evaluation on the provided validation file
Use this section only after selecting models using training-set CV. It reports the held-out UJIIndoorLoc validation file as an additional evaluation, as required by the case brief. No tuning should be performed using these results.

In [ ]:
# IMPORTANT: populate selected_models only after inspecting CV results.
# Example: selected_models = {("classification","BUILDINGID"): fitted_pipeline, ...}
selected_models = {}

def haversine_meters(lat1, lon1, lat2, lon2):
    # Only use if coordinates are geographic degrees. Verify CRS before using.
    R = 6371000.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

if selected_models:
    for (task, target), model in selected_models.items():
        Xv = make_X(validation, strategy="physical", drop_never_detected=True, reference_train=train)
        yv = validation[target]
        pred = model.predict(Xv)
        print(task, target, "validation results")
        if task == "classification":
            print(classification_metrics(yv, pred))
            ConfusionMatrixDisplay.from_predictions(yv,pred,xticks_rotation="vertical")
            plt.tight_layout(); plt.show()
        else:
            print(regression_metrics(yv,pred))
else:
    print("No models selected yet. Select winners based on training CV results before using validation data.")


## 9. Report checklist and limitations
- Report both 80:20 and 70:30 results.
- Include all model comparison tables and mean ± standard deviation from 5-fold CV.
- Explain physical meaning of RSSI and the `+100` sentinel.
- Explain all feature-removal thresholds with a source or empirical justification.
- Include confusion matrices, predicted-vs-actual plots, residual plots, elbow/k-distance/PCA figures, and MLP learning curves.
- Report classification metrics for both `BUILDINGID` and `FLOOR`; regression metrics for both coordinate targets.
- Do not call coordinate errors “meters” until the coordinate reference system is verified. If these are projected coordinates in meters, calculate Euclidean positional error from both predicted coordinates jointly; if geographic degrees, use a geodesic distance.
- Include AI-use disclosure and group contribution table in the final report.
